# Fine-tune Qwen3-4B-Thinking on Meddies Consultant RandomQA (Kaggle 2×T4)

This notebook fine-tunes **Qwen3-4B-Thinking** using [Unsloth](https://unsloth.ai/) on the `Meddies/meddies-consultant` → `RandomQA` subset.

- **Environment:** Kaggle with 2× NVIDIA T4 GPUs
- **Strategy:** DDP (Distributed Data Parallel) via `torchrun` for multi-GPU training
- **Dataset columns:** `question` (Vietnamese clinical question), `answer` (includes `<think>` reasoning)

> **How to run on Kaggle:**
> 1. Upload this notebook to Kaggle
> 2. Set Accelerator to **GPU T4 ×2**
> 3. Enable **Internet**
> 4. Run all cells

## 1. Installation

In [1]:
%%capture
import os, re

# Kaggle / Colab / Local detection
if "KAGGLE_" in "".join(os.environ.keys()):
    # Kaggle environment
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
elif "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Local setup
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Verify GPU Setup

In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {round(props.total_memory / 1024**3, 1)} GB")

assert torch.cuda.device_count() >= 2, "This notebook requires 2 GPUs (Kaggle T4 ×2)!"

PyTorch version: 2.10.0+cu128
CUDA available: True
Number of GPUs: 2
  GPU 0: Tesla T4 — 14.6 GB
  GPU 1: Tesla T4 — 14.6 GB


## 3. Write the DDP Training Script

For multi-GPU DDP training on Kaggle (or any multi-GPU environment), we write a standalone `.py` script and launch it with `torchrun`. This is the recommended approach from [Unsloth DDP docs](https://unsloth.ai/docs/basics/multi-gpu-training-with-unsloth/ddp).

> **Key point:** Unsloth auto-enables DDP when training with >1 GPU via `torchrun`.

In [3]:
%%writefile train_ddp.py
import os, re, glob
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Configuration - FAST TRAINING SETUP
MAX_SEQ_LENGTH = 1024       # Giảm xuống 1024 để train nhanh gấp đôi
LOAD_IN_4BIT   = True
MAX_STEPS      = 600        # Giới hạn 600 bước (đủ hội tụ tốt cho LoRA)
BATCH_SIZE     = 8
GRAD_ACCUM     = 2          # Effective batch size = 8 * 2 * 2(GPUs) = 32
LEARNING_RATE  = 2e-4
PACKING        = True       # Bật packing để không lãng phí GPU cho padding

# Tự động quét tìm checkpoint trong dataset (ví dụ ckpt_dpp_14000)
ckpt_matches = sorted(glob.glob("/kaggle/input/**/checkpoint-*", recursive=True))
RESUME_CHECKPOINT = ckpt_matches[-1] if ckpt_matches else None
print(f"🔄 Resuming from checkpoint: {RESUME_CHECKPOINT}")

# 2. Load Base Model & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-4B-Thinking-2507",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = LOAD_IN_4BIT,
)

# 3. Add LoRA Adapters (Tối ưu 4 module chính để train siêu nhanh)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 4. Load & Format Dataset
dataset = load_dataset("Meddies/meddies-consultant", data_dir="data/randomQA", split="train")

def format_prompts(examples):
    questions = examples["question"]
    answers   = examples["answer"]
    texts = []
    for q, a in zip(questions, answers):
        messages = [
            {"role": "user", "content": q},
            {"role": "assistant", "content": a}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True, num_proc=8, remove_columns=dataset.column_names)

# 5. Training Arguments for DDP
training_args = TrainingArguments(
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_steps                = 20,
    max_steps                   = MAX_STEPS,
    learning_rate               = LEARNING_RATE,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 10,
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "linear",
    seed                        = 3407,
    output_dir                  = "outputs",
    save_strategy               = "steps",
    save_steps                  = 200,
    ddp_find_unused_parameters  = False,
    report_to                   = "none",
)

# 6. Trainer
trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 8,
    packing        = PACKING,
    args           = training_args,
)

# 7. Start Training
if __name__ == "__main__":
    print("🚀 Starting DDP Training...")
    trainer_stats = trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
    print("✅ Training complete!")

    # Only Rank 0 saves the final model
    if int(os.environ.get("LOCAL_RANK", 0)) == 0:
        print("💾 Saving LoRA adapter...")
        model.save_pretrained("qwen3_4b_thinking_meddies_lora")
        tokenizer.save_pretrained("qwen3_4b_thinking_meddies_lora")
        
        print("💾 Saving GGUF (q4_k_m) for deployment...")
        model.save_pretrained_gguf("qwen3_4b_thinking_meddies_gguf", tokenizer, quantization_method="q4_k_m")
        print("🎉 All done! Saved to qwen3_4b_thinking_meddies_gguf/")

Writing train_ddp.py


## 4. Launch DDP Training with `torchrun`

We use `torchrun --nproc_per_node=2` to launch the training script across both T4 GPUs.

This will:
- Spawn 2 training processes (one per GPU)
- Each process gets its own copy of the model
- Data is split across GPUs automatically
- Gradients are synchronized via NCCL

In [4]:
!torchrun --nproc_per_node=2 train_ddp.py

W0704 04:08:46.564000 85 torch/distributed/run.py:852] 
W0704 04:08:46.564000 85 torch/distributed/run.py:852] *****************************************
W0704 04:08:46.564000 85 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0704 04:08:46.564000 85 torch/distributed/run.py:852] *****************************************
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
🔄 Resuming from checkpoint: /kaggle/input/datasets/tu4nhoang/ckpt-dpp-14000/outputs/checkpoint-14000
🔄 Resuming from checkpoint: /kaggle/input/datasets/tu4nhoang/ckpt-dpp-14000/outputs/checkpoint-14000
==((===

## 5. Verify Saved Models

In [5]:
import os

for save_dir in [
    "qwen3_4b_thinking_meddies_lora",
    "qwen3_4b_thinking_meddies_merged",
    "qwen3_4b_thinking_meddies_gguf",
]:
    if os.path.exists(save_dir):
        files = os.listdir(save_dir)
        total_size = sum(
            os.path.getsize(os.path.join(save_dir, f))
            for f in files
            if os.path.isfile(os.path.join(save_dir, f))
        )
        print(f"📁 {save_dir}/")
        print(f"   Files: {len(files)} | Total size: {total_size / 1024**3:.2f} GB")
        for f in sorted(files)[:10]:
            fpath = os.path.join(save_dir, f)
            if os.path.isfile(fpath):
                print(f"   - {f} ({os.path.getsize(fpath) / 1024**2:.1f} MB)")
        if len(files) > 10:
            print(f"   ... and {len(files) - 10} more files")
        print()
    else:
        print(f"⚠️  {save_dir}/ not found")

⚠️  qwen3_4b_thinking_meddies_lora/ not found
⚠️  qwen3_4b_thinking_meddies_merged/ not found
⚠️  qwen3_4b_thinking_meddies_gguf/ not found


## 6. Inference — Test the Fine-tuned Model

Load the LoRA adapter and run inference on a sample Vietnamese clinical question.

In [6]:
import os, glob
from unsloth import FastLanguageModel

# 1. Tự động tìm model đã lưu trong working dir HOẶC checkpoint trong dataset /kaggle/input
model_path = "qwen3_4b_thinking_meddies_lora"
if not os.path.exists(model_path):
    ckpt_matches = sorted(glob.glob("/kaggle/input/**/checkpoint-*", recursive=True))
    if ckpt_matches:
        model_path = ckpt_matches[-1]
    else:
        raise RuntimeError("❌ Không tìm thấy model lora trong /kaggle/working và cũng không tìm thấy checkpoint trong /kaggle/input!")

print(f"📦 Loading model for inference from: {model_path}")

# 2. Load mô hình
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_path,
    max_seq_length = 2048,
    load_in_4bit   = True,
)

# 3. (Tùy chọn) Nếu load từ checkpoint mà chưa có GGUF, xuất GGUF ra tab Output luôn
gguf_dir = "qwen3_4b_thinking_meddies_gguf"
if not os.path.exists(gguf_dir):
    print("⚡ Đang xuất file GGUF ra tab Output (/kaggle/working/)...")
    model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method="q4_k_m")
    print(f"✅ Xuất GGUF thành công vào thư mục: {gguf_dir}/")

# 4. Bật chế độ suy luận nhanh
FastLanguageModel.for_inference(model)
print("🎉 Mô hình đã sẵn sàng cho Inference!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
📦 Loading model for inference from: /kaggle/input/datasets/tu4nhoang/ckpt-dpp-14000/outputs/checkpoint-14000
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.6.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


⚡ Đang xuất file GGUF ra tab Output (/kaggle/working/)...
Unsloth: Merging model weights to 16-bit format...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:22<00:00, 11.19s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:57<00:00, 28.93s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/qwen3_4b_thinking_meddies_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9867-mix-c19e218 (app-b9867-mix-c19e218-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen3_4b_thinking_meddies_gguf_gguf/qwen3-4b-thinking-2507.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Un

In [7]:
# Test questions — Vietnamese clinical scenarios
test_questions = [
    "Bệnh nhân bị đau bụng dữ dội vùng thượng vị kèm nôn, sốt 39 độ C. Cần làm gì để chẩn đoán và xử trí ban đầu?",
    "Giải thích cơ chế tác dụng của Metformin trong điều trị đái tháo đường type 2.",
    "Một bệnh nhân 65 tuổi bị tăng huyết áp kèm đái tháo đường, nên chọn nhóm thuốc hạ áp nào?",
]

for i, question in enumerate(test_questions):
    print(f"\n{'='*80}")
    print(f"Question {i+1}: {question}")
    print(f"{'='*80}")
    
    messages = [{"role": "user", "content": question}]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=1024,
        temperature=0.6,
        top_p=0.95,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nAnswer:\n{response}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Question 1: Bệnh nhân bị đau bụng dữ dội vùng thượng vị kèm nôn, sốt 39 độ C. Cần làm gì để chẩn đoán và xử trí ban đầu?

Answer:
 Tôi cần phân tích câu hỏi của người dùng. Người dùng mô tả một bệnh nhân có triệu chứng: đau bụng dữ dội vùng thượng vị, kèm theo nôn và sốt 39 độ C. Đây là một tình huống lâm sàng cấp tính cần được xử trí nhanh chóng và chính xác. **Bước 1: Phân tích triệu chứng và nguy cơ** - **Đau bụng dữ dội vùng thượng vị**: Đây là dấu hiệu quan trọng. "Vùng thượng vị" (epigastric) thường gợi ý các bệnh lý ở dạ dày, tá tràng, hoặc trung tâm dạ dày - ruột. - **Nôn**: Có thể do phản xạ hoặc do bệnh lý nền. - **Sốt 39 độ C**: Đây là dấu hiệu cảnh báo nhiễm trùng toàn thân. **Bước 2: Xác định các chẩn đoán cần loại trừ** Với các triệu chứng này, cần cân nhắc các bệnh lý cấp tính nguy hiểm sau: 1. **Viêm loét dạ dày tá tràng cấp tính**: Có thể do H. pylori hoặc NSAID. 2. **Đau dạ dày cấp tính**: Có thể do trào ngược dạ dày thực quản nặng hoặc viêm thực quản. 3. **Tắc ruột 

## 7. (Optional) Push to Hugging Face Hub

Uncomment and fill in your token/username to push the model.

In [8]:
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

# # Push LoRA adapters
# model.push_to_hub("YOUR_USERNAME/qwen3-4b-thinking-meddies-lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("YOUR_USERNAME/qwen3-4b-thinking-meddies-lora", token="YOUR_HF_TOKEN")

# # Push merged model
# model.push_to_hub_merged(
#     "YOUR_USERNAME/qwen3-4b-thinking-meddies-merged",
#     tokenizer,
#     save_method="merged_16bit",
#     token="YOUR_HF_TOKEN",
# )

# # Push GGUF
# model.push_to_hub_gguf(
#     "YOUR_USERNAME/qwen3-4b-thinking-meddies-gguf",
#     tokenizer,
#     quantization_method=["q4_k_m", "q8_0"],
#     token="YOUR_HF_TOKEN",
# )

## 8. Memory Stats

In [9]:
import torch

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    max_mem = round(torch.cuda.max_memory_reserved(i) / 1024**3, 3)
    total_mem = round(props.total_memory / 1024**3, 3)
    print(f"GPU {i} ({props.name}): {max_mem} / {total_mem} GB reserved")

GPU 0 (Tesla T4): 4.207 / 14.562 GB reserved
GPU 1 (Tesla T4): 0.146 / 14.562 GB reserved
